# Pre-M0.5 — Loading and Inspecting IQ Files

## Unit Objective

In this notebook you will learn how to **create, save, load, and inspect IQ signal arrays** stored as `.npz` files. By the end you will be able to:

- Safely write and read NPZ files using `Path` and `np.load`.
- Inspect every relevant property of an IQ array: shape, ndim, dtype, nbytes, and axis semantics.
- Diagnose and correct common loading errors: nonexistent files, wrong keys, wrong ndim, and axis swaps.
- Verify that power computed from real/imaginary components agrees with power computed from the complex representation.

## What You Should Learn

1. The canonical IQ layout: `X.shape == (N, 2, L)` where axis 0 = examples, axis 1 = I/Q (0 = I, 1 = Q), axis 2 = time samples.
2. How `Path.exists()` guards against missing files and how `allow_pickle=False` prevents security issues.
3. How to implement a reusable `inspect_iq_array()` function.
4. How to detect and fix an axis-swap error — a subtle bug that can silently corrupt every downstream computation.
5. How to verify power agreement between real/imaginary decomposition and complex absolute-value computation.

## IQ Data Contract

All notebooks in this series share the same canonical layout:

```
X.shape == (N, 2, L)
    axis 0 → examples (N)
    axis 1 → I/Q      (0 = I, 1 = Q)
    axis 2 → time samples (L)
dtype = np.float32
SEED  = 42
```

Every function and validation in this notebook assumes this convention.

---
## 1 — Setup and Imports

In [ ]:
import numpy as np
from pathlib import Path

SEED = 42
rng = np.random.default_rng(SEED)

DEMO_PATH = Path("demo_iq.npz")

print(f"NumPy version : {np.__version__}")
print(f"DEMO_PATH     : {DEMO_PATH.resolve()}")

---
## 2 — Creating a Demo NPZ File

Before we can practice loading, we need a file to load. The cell below creates a small but fully valid IQ array and saves it to disk.

Remember: the canonical shape is `(N, 2, L)` with dtype `float32`.

In [ ]:
N, L = 10, 200

I = rng.standard_normal((N, L)).astype(np.float32)
Q = rng.standard_normal((N, L)).astype(np.float32)

X_demo = np.stack([I, Q], axis=1)  # shape (N, 2, L)

assert X_demo.shape == (N, 2, L)
assert X_demo.dtype == np.float32

np.savez(DEMO_PATH, iq=X_demo)
print(f"Saved {DEMO_PATH}  — shape: {X_demo.shape}, dtype: {X_demo.dtype}")

---
## 3 — Safe Loading with Path and np.load

### 3.1 Check existence before loading

In [ ]:
print(f"File exists: {DEMO_PATH.exists()}")

### 3.2 Load with `allow_pickle=False`

Using `allow_pickle=False` is a security best practice — it prevents arbitrary code execution from malformed NPZ files.

In [ ]:
data = np.load(DEMO_PATH, allow_pickle=False)
print(f"NPZ keys : {list(data.keys())}")

iq = data["iq"]
print(f"iq.shape : {iq.shape}")
print(f"iq.ndim  : {iq.ndim}")
print(f"iq.dtype : {iq.dtype}")
print(f"iq.nbytes: {iq.nbytes}")

### 3.3 Confirm we recovered the original data

In [ ]:
assert np.array_equal(iq, X_demo), "Loaded data does not match original!"
print("Round-trip save/load verified.")

---
## 4 — Implementing `inspect_iq_array`

A good inspection function should:
1. Verify the array is NumPy.
2. Check ndim == 3.
3. Check axis 1 has exactly 2 elements (I and Q).
4. Print shape, dtype, nbytes, and axis labels.
5. Return a summary dictionary.

In [ ]:
def inspect_iq_array(iq):
    """Inspect a loaded IQ array and return a summary dictionary.
    
    Expected canonical layout:
        iq.shape == (N, 2, L)
        axis 0 -> examples (N)
        axis 1 -> I/Q      (0=I, 1=Q)
        axis 2 -> time samples (L)
        dtype   -> float32
    """
    issues = []
    
    if not isinstance(iq, np.ndarray):
        issues.append(f"Expected np.ndarray, got {type(iq).__name__}")
        return {"ok": False, "issues": issues}
    
    if iq.ndim != 3:
        issues.append(f"Expected ndim=3, got {iq.ndim}")
    
    if iq.ndim >= 2 and iq.shape[1] != 2:
        issues.append(f"Expected axis 1 == 2 (I/Q), got {iq.shape[1]}")
    
    if iq.dtype != np.float32:
        issues.append(f"Expected dtype=float32, got {iq.dtype}")
    
    N = iq.shape[0] if iq.ndim >= 1 else "?"
    L = iq.shape[2] if iq.ndim == 3 else "?"
    
    summary = {
        "ok": len(issues) == 0,
        "shape": iq.shape,
        "ndim": iq.ndim,
        "dtype": iq.dtype,
        "nbytes": iq.nbytes,
        "N": N,
        "L": L,
        "issues": issues,
    }
    
    print(f"Shape  : {iq.shape}")
    print(f"ndim   : {iq.ndim}")
    print(f"dtype  : {iq.dtype}")
    print(f"nbytes : {iq.nbytes}")
    print(f"N (examples) : {N}")
    print(f"L (samples)  : {L}")
    print(f"Issues found : {len(issues)}")
    for issue in issues:
        print(f"  - {issue}")
    
    return summary

result = inspect_iq_array(iq)
print(f"\nAll checks passed: {result['ok']}")

---
## 5 — Pedagogical Error Examples

The function above catches problems, but let us practice diagnosing errors by hand first.

### 5.1 Nonexistent file

In [ ]:
missing_path = Path("this_file_does_not_exist.npz")

print(f"File exists: {missing_path.exists()}")

if not missing_path.exists():
    print("Caught it: file does not exist. Cannot load.")

### 5.2 Wrong key

In [ ]:
data = np.load(DEMO_PATH, allow_pickle=False)
print(f"Available keys: {list(data.keys())}")

try:
    bad = data["wrong_key"]
except KeyError as e:
    print(f"KeyError caught: {e}")
    print("The file does not contain a 'wrong_key' entry.")
finally:
    data.close()

### 5.3 Wrong ndim — loading a 2D array instead of 3D

In [ ]:
bad_ndim = rng.standard_normal((2, 200)).astype(np.float32)
print(f"Shape: {bad_ndim.shape}, ndim: {bad_ndim.ndim}")

result_bad = inspect_iq_array(bad_ndim)
print(f"All checks passed: {result_bad['ok']}")

### 5.4 Unexpected IQ axis — axis 1 has 3 elements instead of 2

In [ ]:
bad_iq_axis = rng.standard_normal((10, 3, 200)).astype(np.float32)
print(f"Shape: {bad_iq_axis.shape}, ndim: {bad_iq_axis.ndim}")

result_bad_axis = inspect_iq_array(bad_iq_axis)
print(f"All checks passed: {result_bad_axis['ok']}")

---
## 6 — Deliberate Axis Swap — A Subtle and Dangerous Bug

An axis swap is one of the most insidious bugs in signal processing. The array still has 3 dimensions, the dtype is still `float32`, and it even *looks* reasonable — but the I and Q values are scrambled across time.

The cell below creates the swapped version deliberately.

In [ ]:
X_swapped = np.transpose(X_demo, (0, 2, 1))  # shape (N, L, 2) — WRONG

print(f"Original shape : {X_demo.shape}")
print(f"Swapped shape  : {X_swapped.shape}")
print(f"Swapped ndim   : {X_swapped.ndim}")
print(f"Swapped dtype  : {X_swapped.dtype}")
print()
print("At first glance both look plausible — but the semantics are completely wrong.")
print("In the swapped version, axis 1 is time samples and axis 2 is I/Q.")
print("Any function expecting (N, 2, L) will misinterpret the data.")

---
## 7 — Student Exercises

Complete the tasks below before moving to the Pass Criterion section.

### Exercise 1 — Load and inspect

Load `demo_iq.npz` and run `inspect_iq_array` on the result. Confirm all checks pass.

In [ ]:
# STUDENT ATTEMPT
# TODO: Load demo_iq.npz and call inspect_iq_array

# data = np.load(...)
# iq_loaded = data[...]
# result = inspect_iq_array(...)
# print(f"All checks passed: {result['ok']}")


### OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT

```python
data = np.load(DEMO_PATH, allow_pickle=False)
iq_loaded = data["iq"]
result = inspect_iq_array(iq_loaded)
print(f"All checks passed: {result['ok']}")
```

### Exercise 2 — Diagnose the axis swap

The variable `X_swapped` was loaded above. Use its `.shape` to determine which axis holds I/Q and which holds time samples. Then write the correction to recover `X_fixed`.

In [ ]:
# STUDENT ATTEMPT
# TODO: Inspect X_swapped.shape, identify the problem, and fix it

# print(f"X_swapped.shape = {X_swapped.shape}")
# X_fixed = ...  # your correction
# assert np.array_equal(X_fixed, X_demo)


### OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT

```python
print(f"X_swapped.shape = {X_swapped.shape}")
# shape is (N, L, 2) — axis 1 is time, axis 2 is I/Q. Wrong.
X_fixed = np.transpose(X_swapped, (0, 2, 1))
assert np.array_equal(X_fixed, X_demo)
print("Axis swap corrected.")
```

### Exercise 3 — Power agreement

Given I = `iq["iq"][:, 0, :]` and Q = `iq["iq"][:, 1, :]`, compute power two independent ways and compare.

In [ ]:
# STUDENT ATTEMPT
# TODO: Compute P_iq from I and Q, then P_complex from the complex form

# I = ...
# Q = ...
# P_iq = ...
# z = ...
# P_complex = ...
# print(f"P_iq      = {P_iq}")
# print(f"P_complex = {P_complex}")
# print(f"Agree: {np.allclose(P_iq, P_complex, rtol=1e-5, atol=1e-7)}")


### OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT

```python
I = iq_loaded[:, 0, :]
Q = iq_loaded[:, 1, :]

P_iq = np.mean(I**2 + Q**2)

z = I + 1j * Q
P_complex = np.mean(np.abs(z)**2)

print(f"P_iq      = {P_iq}")
print(f"P_complex = {P_complex}")
print(f"Agree: {np.allclose(P_iq, P_complex, rtol=1e-5, atol=1e-7)}")
```

---
## 8 — Pass Criterion Challenge

All three gates must pass for a final status of **PASS**. If any gate fails, the status is **WAIT**.

---
### PC-1 — Independent Axis Explanation

Answer the six questions below in the Markdown cell that follows.

### STUDENT AXIS EXPLANATION

Answer each question in your own words.

**Q1.** What does axis 0 represent in the canonical IQ layout `(N, 2, L)`?

YOUR ANSWER HERE

**Q2.** What does axis 1 represent, and what do the values 0 and 1 on that axis mean?

YOUR ANSWER HERE

**Q3.** What does axis 2 represent?

YOUR ANSWER HERE

**Q4.** If `X.shape == (100, 2, 500)`, how many examples are there, and how many time samples per example?

YOUR ANSWER HERE

**Q5.** Why is `X[:, 0, :]` the in-phase (I) component and `X[:, 1, :]` the quadrature (Q) component?

YOUR ANSWER HERE

**Q6.** If someone gives you an array with shape `(100, 500, 2)`, what is wrong with it and why?

YOUR ANSWER HERE

In [ ]:
AXES_EXPLANATION_VERIFIED = False  # Student/instructor: change to True after manual review

# This flag must be True for PC-1 to pass.
# It requires reading the student's written answers above and confirming correctness.

---
### PC-2 — Injected Axis Swap Recovery

The variable `X_swapped` (created in Section 6) has shape `(N, L, 2)` — a deliberate axis-swap error.

Your task:
1. Inspect its shape.
2. Identify which axis holds I/Q and which holds time samples.
3. Explain why this is wrong.
4. Write the correction and store it in `X_fixed`.
5. Verify that `X_fixed` exactly matches the original `X_demo`.

In [ ]:
# STUDENT ATTEMPT — PC-2
# TODO: Diagnose X_swapped and produce X_fixed

# print(f"X_swapped.shape = {X_swapped.shape}")
# TODO: identify the problem
# TODO: write correction
# X_fixed = ...


### OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT

```python
print(f"X_swapped.shape = {X_swapped.shape}")
# Shape is (N, L, 2) — axis 1 is time samples, axis 2 is I/Q.
# The canonical layout requires axis 1 = I/Q and axis 2 = time samples.
# Therefore we need to swap axes 1 and 2.

X_fixed = np.transpose(X_swapped, (0, 2, 1))

assert X_fixed.shape == X_demo.shape, f"Shape mismatch: {X_fixed.shape} != {X_demo.shape}"
assert X_fixed.dtype == X_demo.dtype, f"Dtype mismatch: {X_fixed.dtype} != {X_demo.dtype}"
assert np.array_equal(X_fixed, X_demo), "Values do not match!"
print("Axis swap corrected and verified.")
```

In [ ]:
# === PC-2 VALIDATION ===
# This cell verifies your X_fixed is correct.
# Run this AFTER you have defined X_fixed above.

# Uncomment the lines below after you have defined X_fixed:
# axis_swap_corrected = (
#     X_fixed.shape == X.shape
#     and X_fixed.dtype == X.dtype
#     and np.array_equal(X_fixed, X)
# )
# assert X_fixed.shape == X.shape
# assert X_fixed.dtype == X.dtype
# assert np.array_equal(X_fixed, X)
# print("PC-2 PASSED: axis swap correctly recovered.")

---
### PC-3 — IQ Power Agreement

Compute the mean power of the IQ signal using two independent methods:

**Method A (real/imaginary):**  `P_iq = np.mean(I**2 + Q**2)`

**Method B (complex):**  `z = I + 1j*Q;  P_complex = np.mean(np.abs(z)**2)`

These must agree within tolerance.

In [ ]:
# STUDENT ATTEMPT — PC-3
# TODO: Implement both power computations

# I = iq_loaded[:, 0, :]
# Q = iq_loaded[:, 1, :]

# P_iq = ...
# z = ...
# P_complex = ...


### OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT

```python
I = iq_loaded[:, 0, :]
Q = iq_loaded[:, 1, :]

P_iq = np.mean(I**2 + Q**2)

z = I + 1j * Q
P_complex = np.mean(np.abs(z)**2)

print(f"P_iq      = {P_iq}")
print(f"P_complex = {P_complex}")
```

In [ ]:
# === PC-3 VALIDATION ===
# Uncomment after defining P_iq and P_complex above.

POWER_RTOL = 1e-5
POWER_ATOL = 1e-7

# power_consistency = np.allclose(P_iq, P_complex, rtol=POWER_RTOL, atol=POWER_ATOL)
# assert power_consistency, f"Power mismatch: P_iq={P_iq}, P_complex={P_complex}"
# print("PC-3 PASSED: power agreement verified.")

---
## 9 — Automatic Validations

Run the cell below to check all objective gates at once. This only verifies the **code-level** criteria — PC-1 also requires manual review of your written answers.

In [ ]:
print("=" * 60)
print("PRE-M0.5 AUTOMATIC VALIDATIONS")
print("=" * 60)

# --- PC-1 ---
pc1_pass = AXES_EXPLANATION_VERIFIED is True
print(f"\nPC-1 (Axis Explanation) : {'PASS' if pc1_pass else 'WAIT'} (requires manual review)")

# --- PC-2 ---
try:
    assert X_fixed.shape == X_demo.shape
    assert X_fixed.dtype == X_demo.dtype
    assert np.array_equal(X_fixed, X_demo)
    pc2_pass = True
except Exception as e:
    pc2_pass = False
    print(f"  PC-2 error: {e}")
print(f"PC-2 (Axis Swap Fix)    : {'PASS' if pc2_pass else 'WAIT'}")

# --- PC-3 ---
try:
    assert np.allclose(P_iq, P_complex, rtol=POWER_RTOL, atol=POWER_ATOL)
    pc3_pass = True
except Exception as e:
    pc3_pass = False
    print(f"  PC-3 error: {e}")
print(f"PC-3 (Power Agreement)  : {'PASS' if pc3_pass else 'WAIT'}")

print("\n" + "=" * 60)
if pc1_pass and pc2_pass and pc3_pass:
    print("PRE-M0.5 FINAL STATUS: PASS")
else:
    print("PRE-M0.5 FINAL STATUS: WAIT")
print("=" * 60)

---
## 10 — Manual Evaluation of Axis Explanation

The automatic validation above cannot grade your written answers. A human reviewer (or you, self-assessing) must read the six answers in the **STUDENT AXIS EXPLANATION** section and confirm they are correct.

Reference answers:

| Q | Expected answer |
|---|------------------|
| Q1 | Axis 0 represents the **examples** (individual IQ records or observations). |
| Q2 | Axis 1 represents **I/Q channels**: 0 = In-phase (I), 1 = Quadrature (Q). |
| Q3 | Axis 2 represents **time samples** within each example. |
| Q4 | 100 examples, 500 time samples per example. |
| Q5 | Because axis 1 index 0 selects the first channel (I) and index 1 selects the second channel (Q) by convention. |
| Q6 | Shape `(100, 500, 2)` puts I/Q on axis 2 and time on axis 1 — the axes are swapped relative to the canonical layout. |

Once verified, set `AXES_EXPLANATION_VERIFIED = True` in the PC-1 code cell.

---
## 11 — Final Report

Re-run the validation cell in Section 9 after completing all three gates. Your final status should read:

```
PRE-M0.5 FINAL STATUS: PASS
```

If it reads **WAIT**, revisit the gate that failed and try again.

### Checklist
- [ ] PC-1: Six axis questions answered and verified.
- [ ] PC-2: `X_fixed` recovers `X_demo` exactly from `X_swapped`.
- [ ] PC-3: `P_iq` and `P_complex` agree within tolerance.

In [ ]:
# Cleanup (optional)
if DEMO_PATH.exists():
    DEMO_PATH.unlink()
    print(f"Cleaned up {DEMO_PATH}")
else:
    print(f"{DEMO_PATH} not found — nothing to clean up.")